In [1]:
import os
import smtplib
import datetime
import shutil
import re
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# 1. Install & Import
try:
    import gradio as gr
    import whisper
    from google.colab import drive, ai
except ImportError:
    !pip install -q gradio openai-whisper
    import gradio as gr
    import whisper
    from google.colab import drive, ai

# 2. Setup Drive & Whisper
drive.mount('/content/drive', force_remount=True)
SAVE_DIR = "/content/drive/MyDrive/Meeting_Notes/"
os.makedirs(SAVE_DIR, exist_ok=True)
whisper_model = whisper.load_model("base")

def process_meeting(audio_path):
    if not audio_path:
        return "☑ Please record audio or upload a file first."

    ts = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")

    # B. Transcription
    print(f"[{ts}] Transcribing...")
    transcription_result = whisper_model.transcribe(audio_path)
    transcript = transcription_result['text']

    # C. AI Extraction via Colab's native AI module (No API Key needed)
    print(f"[{ts}] Extracting data via Colab AI...")

    full_prompt = f"""
    You are a professional sales coordinator. Extract information strictly from the provided transcript.
    If a specific field is not mentioned, respond with 'Not mentioned'.

    FIELDS TO EXTRACT:
    - Customer Name:
    - Company:
    - Attendees:
    - Key Requests:
    - Planned Response and Timeline:
    - Meeting Assessment:
    - Summary (3-4 sentences):

    TRANSCRIPT:
    {transcript}
    """

    try:
        # Using the built-in Colab AI proxy
        structured_note = ai.generate_text(prompt=full_prompt)
    except Exception as e:
        structured_note = f"AI Extraction Error: {str(e)}"

    # D. Save to Google Drive
    final_audio_name = f"{ts}.m4a"
    final_text_name = f"{ts}.txt"

    shutil.copy(audio_path, os.path.join(SAVE_DIR, final_audio_name))
    with open(os.path.join(SAVE_DIR, final_text_name), "w") as f:
        f.write(f"MEETING DATE: {ts}\n\nAI SUMMARY:\n{structured_note}\n\n--- FULL TRANSCRIPT ---\n{transcript}")

    # E. Send Email
    email_status = send_email(structured_note, transcript, ts)

    return f"✅ SUCCESS\n\nDrive Folder: Meeting_Notes/{final_audio_name}\nEmail: {email_status}\n\n--- EXTRACTED DATA ---\n{structured_note}"

def send_email(note, transcript, ts):
    SENDER = "notifactorystat@gmail.com"
    APP_PASSWORD = "wbwsnvqqswxpclfs"
    RECEIVER = "michel.song@gmail.com"

    msg = MIMEMultipart()
    msg['From'] = SENDER
    msg['To'] = RECEIVER
    msg['Subject'] = f"Meeting Record: {ts}"

    body = f"Automated Meeting Summary:\n\n{note}\n\n--- FULL TRANSCRIPT ---\n{transcript}"
    msg.attach(MIMEText(body, 'plain'))

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(SENDER, APP_PASSWORD)
            server.send_message(msg)
        return "Sent Successfully"
    except Exception as e:
        return f"Email Failed: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("#    Field Sales Audio Sync")
    gr.Markdown("Record or upload your post-meeting summary. AI will transcribe, extract, save to Drive, and email.")

    with gr.Tabs():
        with gr.TabItem("Tab 1: Microphone"):
            mic_input = gr.Audio(sources=["microphone"], type="filepath", label="Tap to Record")
            mic_btn = gr.Button("Submit Recording", variant="primary")

        with gr.TabItem("Tab 2: File Upload"):
            file_input = gr.Audio(sources=["upload"], type="filepath", label="Upload Fallback")
            file_btn = gr.Button("Submit File", variant="primary")

    output = gr.Textbox(label="Result & AI Summary", lines=12)

    mic_btn.click(process_meeting, inputs=mic_input, outputs=output)
    file_btn.click(process_meeting, inputs=file_input, outputs=output)

demo.queue().launch(share=True, debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 45.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 348MiB/s]
/tmp/ipykernel_2289/2033596630.py:97: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e6f672ee63401c55bd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[2026-04-16_0720] Transcribing...
[2026-04-16_0720] Extracting data via Colab AI...
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e6f672ee63401c55bd.gradio.live
